# RAG vs. No-RAG — Multi-Family LLM Evaluation
## Notebook 03 — Vector DB: Corpus B (RRiF PDF, Croatian)

**Corpus B** — Official Croatian Chart of Accounts (RRiF, XXVIII. edition, 2025).
- ~2,461 hierarchical documents (Croatian names + parent context)
- Built with **three embedding models** (ablation study):

| Collection | Embedding model | Type |
|------------|-----------------|------|
| `corpus_b_nomic`  | `nomic-embed-text`     | English-optimised (cross-lingual gap) |
| `corpus_b_qwen3s` | `qwen3-embedding:0.6b` | Multilingual lightweight |
| `corpus_b_qwen3m` | `qwen3-embedding:4b`   | Multilingual mid-size |

**Expected runtime:** ~30–60 min per collection depending on GPU.
Checkpoints are implicit (ChromaDB persists to disk).


---
## ⚙️ Configuration

In [6]:
# ── CONFIGURATION — edit only this cell ──────────────────────────────────────

OLLAMA_BASE_URL = "http://100.119.36.113:11434"   # ← set your server IP

DATA_DIR    = "data"
CHROMA_DIR  = f"{DATA_DIR}/chroma_db"
RESULTS_DIR = f"{DATA_DIR}/results"
FIGURES_DIR = f"{DATA_DIR}/figures"

TOP_K       = 5
TEMPERATURE = 0
MAX_TOKENS  = 512

# ── Embedding models ───────────────────────────────────────────────────────────
EMBED_MODELS = {
    "nomic":   "nomic-embed-text",       # English-optimised
    "qwen3s":  "qwen3-embedding:0.6b",   # Multilingual lightweight
    "qwen3m":  "qwen3-embedding:4b",     # Multilingual mid-size
}

# ── ChromaDB collections ───────────────────────────────────────────────────────
COLLECTIONS = {
    "rag_a":        ("corpus_a_nomic",    "nomic"),
    "rag_b_nomic":  ("corpus_b_nomic",    "nomic"),
    "rag_b_qwen3s": ("corpus_b_qwen3s",   "qwen3s"),
    "rag_b_qwen3m": ("corpus_b_qwen3m",   "qwen3m"),
}
# Format: condition → (collection_name, embed_key)

CONDITIONS = ["no_rag", "rag_a", "rag_b_nomic", "rag_b_qwen3s", "rag_b_qwen3m"]

COND_LABELS = {
    "no_rag":        "No RAG",
    "rag_a":         "RAG-A (EN, nomic)",
    "rag_b_nomic":   "RAG-B (HR, nomic)",
    "rag_b_qwen3s":  "RAG-B (HR, qwen3-0.6b)",
    "rag_b_qwen3m":  "RAG-B (HR, qwen3-4b)",
}

# ── Model families ─────────────────────────────────────────────────────────────
MODELS = {
    "Qwen3.5": [
        "qwen3.5:0.8b", "qwen3.5:2b", "qwen3.5:4b", "qwen3.5:9b",
        "qwen3.5:27b",  "qwen3.5:35b", "qwen3.5:122b",
    ],
    "Llama": [
        "llama3.2:1b", "llama3.2:3b", "llama3.1:8b", "llama3.3:70b",
    ],
    "Gemma": [
        "gemma3:1b", "gemma3:4b", "gemma3:12b", "gemma3:27b",
    ],
    "Phi": [
        "phi4-mini", "phi4",
    ],
    "DeepSeek": [
        "deepseek-r1:1.5b", "deepseek-r1:7b", "deepseek-r1:14b",
        "deepseek-r1:32b",  "deepseek-r1:70b",
    ],
}
ALL_MODELS = [m for family in MODELS.values() for m in family]

MODEL_PARAMS = {
    "qwen3.5:0.8b":0.8,  "qwen3.5:2b":2,    "qwen3.5:4b":4,
    "qwen3.5:9b":9,      "qwen3.5:27b":27,   "qwen3.5:35b":35,
    "qwen3.5:122b":122,
    "llama3.2:1b":1,     "llama3.2:3b":3,    "llama3.1:8b":8,
    "llama3.3:70b":70,
    "gemma3:1b":1,       "gemma3:4b":4,      "gemma3:12b":12,
    "gemma3:27b":27,
    "phi4-mini":3.8,     "phi4":14,
    "deepseek-r1:1.5b":1.5, "deepseek-r1:7b":7,  "deepseek-r1:14b":14,
    "deepseek-r1:32b":32,   "deepseek-r1:70b":70,
}

import os
for d in [DATA_DIR, CHROMA_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

total_calls = len(ALL_MODELS) * len(CONDITIONS) * 1060
print("✓ Configuration loaded")
print(f"  Models:      {len(ALL_MODELS)}  ({', '.join(MODELS.keys())})")
print(f"  Conditions:  {len(CONDITIONS)}")
print(f"  Questions:   1,060")
print(f"  Total calls: {total_calls:,}")
print()
print("  Embedding models:")
for k, v in EMBED_MODELS.items():
    print(f"    {k:8s}: {v}")


✓ Configuration loaded
  Models:      22  (Qwen3.5, Llama, Gemma, Phi, DeepSeek)
  Conditions:  5
  Questions:   1,060
  Total calls: 116,600

  Embedding models:
    nomic   : nomic-embed-text
    qwen3s  : qwen3-embedding:0.6b
    qwen3m  : qwen3-embedding:4b


---
## 📦 Dependencies

In [7]:
import subprocess
subprocess.run(
    ["pip", "install", "-q", "--root-user-action=ignore", "chromadb", "requests", "tqdm"],
    capture_output=True
)
print("✓ OK")


✓ OK


---
## 🔌 Check Ollama & embedding models

In [8]:
import requests, time, json

ok_models = []
r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
available = [m["name"] for m in r.json().get("models", [])]

print("Embedding model status:")
for key, model in EMBED_MODELS.items():
    ok = any(model in m for m in available)
    status = "✓" if ok else "✗ MISSING"
    if ok: ok_models.append(key)
    print(f"  {status}  [{key}]  {model}")
    if not ok:
        print(f"       Run: docker exec -it ollama ollama pull {model}")

print(f"\nReady to build: {ok_models}")


Embedding model status:
  ✓  [nomic]  nomic-embed-text
  ✓  [qwen3s]  qwen3-embedding:0.6b
  ✓  [qwen3m]  qwen3-embedding:4b

Ready to build: ['nomic', 'qwen3s', 'qwen3m']


---
## 📄 Load Corpus B documents

In [9]:
with open(f"{DATA_DIR}/corpus_b_rrif_documents.json", encoding="utf-8") as f:
    corpus_b_docs = json.load(f)

from collections import Counter
level_counts = Counter(d["razina"] for d in corpus_b_docs)
print(f"✓ Corpus B: {len(corpus_b_docs)} documents")
for lvl, n in sorted(level_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {lvl:15s}: {n}")
print()
print("Example:")
sin = [d for d in corpus_b_docs if d["razina"] == "sinteticki"]
print(sin[0]["text"])


✓ Corpus B: 2462 documents
  analiticki     : 1546
  sinteticki     : 527
  subanaliticki  : 300
  skupina        : 79
  razred         : 10

Example:
Konto: 000
Naziv: Potraživanja za upisani a neuplaćeni dionički kapital (analitika po upisnicima)
Razina: sinteticki
Skupina: 00 — POTRAŽIVANJA ZA UPISANI A NEUPLAĆENI KAPITAL
Razred: 0 — POTRAŽIVANJA ZA UPISANI KAPITAL I DUGOTRAJNA


In [10]:
with open(f"{DATA_DIR}/corpus_b_rrif_documents.json", encoding="utf-8") as f:
    corpus_b_docs = json.load(f)

# Deduplikacija (već riješeno u 01, ali za sigurnost)
seen  = set()
unique_docs = []
for doc in corpus_b_docs:
    if doc["id"] not in seen:
        seen.add(doc["id"])
        unique_docs.append(doc)
corpus_b_docs = unique_docs

print(f"✓ Corpus B: {len(corpus_b_docs)} dokumenata (deduplicirani)")

✓ Corpus B: 2462 dokumenata (deduplicirani)


---
## 🗄️ Build all three Corpus B collections

In [11]:
import chromadb
from tqdm import tqdm

CORPUS_B_COLLECTIONS = {
    "nomic":   "corpus_b_nomic",
    "qwen3s":  "corpus_b_qwen3s",
    "qwen3m":  "corpus_b_qwen3m",
}

def get_embedding(text, model):
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=60
    )
    r.raise_for_status()
    return r.json()["embedding"]

client     = chromadb.PersistentClient(path=CHROMA_DIR)
BATCH_SIZE = 50

for embed_key, coll_name in CORPUS_B_COLLECTIONS.items():
    embed_model = EMBED_MODELS[embed_key]

    # Skip if model not available
    if embed_key not in ok_models:
        print(f"\n⚠ Skipping '{embed_key}' — model not available")
        continue

    existing = [c.name for c in client.list_collections()]
    print(f"\n{'='*62}")
    print(f"  Building: {coll_name}")
    print(f"  Model:    {embed_model}")
    print(f"{'='*62}")

    if coll_name in existing:
        coll = client.get_collection(coll_name)
        if coll.count() == len(corpus_b_docs):
            print(f"  ✓ Already complete ({coll.count()} docs) — skipping.")
            continue
        else:
            client.delete_collection(coll_name)

    coll   = client.create_collection(coll_name, metadata={"hnsw:space": "cosine"})
    errors = []
    t0     = time.time()

    for i in tqdm(range(0, len(corpus_b_docs), BATCH_SIZE),
                  desc=f"  {embed_key}", ncols=70):
        batch = corpus_b_docs[i:i+BATCH_SIZE]
        ids, embeddings, documents, metadatas = [], [], [], []
        for doc in batch:
            try:
                ids.append(doc["id"])
                embeddings.append(get_embedding(doc["text"], embed_model))
                documents.append(doc["text"])
                metadatas.append(doc["metadata"])
            except Exception as e:
                errors.append({"id": doc["id"], "error": str(e)})
        if ids:
            coll.add(ids=ids, embeddings=embeddings,
                     documents=documents, metadatas=metadatas)

    elapsed = time.time() - t0
    print(f"  ✓ Done in {elapsed:.0f}s  "
          f"({elapsed/len(corpus_b_docs):.2f}s/doc)  "
          f"Docs: {coll.count()}  Errors: {len(errors)}")



  Building: corpus_b_nomic
  Model:    nomic-embed-text


  nomic: 100%|████████████████████████| 50/50 [01:17<00:00,  1.55s/it]


  ✓ Done in 77s  (0.03s/doc)  Docs: 2462  Errors: 0

  Building: corpus_b_qwen3s
  Model:    qwen3-embedding:0.6b


  qwen3s: 100%|███████████████████████| 50/50 [06:40<00:00,  8.01s/it]


  ✓ Done in 401s  (0.16s/doc)  Docs: 2462  Errors: 0

  Building: corpus_b_qwen3m
  Model:    qwen3-embedding:4b


  qwen3m: 100%|███████████████████████| 50/50 [27:08<00:00, 32.58s/it]

  ✓ Done in 1629s  (0.66s/doc)  Docs: 2462  Errors: 0


---
## 🔍 Test retrieval — all three embedding models

In [12]:
def retrieve_b(query, embed_key, k=TOP_K):
    coll_name   = CORPUS_B_COLLECTIONS[embed_key]
    embed_model = EMBED_MODELS[embed_key]
    coll        = client.get_collection(coll_name)
    vec         = get_embedding(query, embed_model)
    res         = coll.query(query_embeddings=[vec], n_results=k,
                             include=["metadatas","distances"])
    return [{"kljuc":      m.get("kljuc",""),
             "naziv":      m.get("naziv","")[:50],
             "similarity": round(1-d, 4)}
            for m, d in zip(res["metadatas"][0], res["distances"][0])]

test_queries = [
    "What is the account code for Land?",
    "What is the account code for Cash in bank?",
    "What is the account description for 020?",
]

for query in test_queries:
    print(f"QUERY: '{query}'")
    for key in ok_models:
        r = retrieve_b(query, key)
        print(f"  [{key:7s}] top-1: "
              f"{r[0]['kljuc']:>6}  "
              f"sim={r[0]['similarity']:.3f}  "
              f"{r[0]['naziv'][:40]}")
    print()

print("✓ Corpus B ready")
print("Next: 04_evaluate.ipynb")


QUERY: 'What is the account code for Land?'
  [nomic  ] top-1:   2229  sim=0.545  Ostali dobavljači – fizičke osobe
  [qwen3s ] top-1:  11429  sim=0.484  Diskont kod ulaganja u komercijale zapis
  [qwen3m ] top-1:   4080  sim=0.547  Trošak zemljišta za tržišnu izgradnju

QUERY: 'What is the account code for Cash in bank?'
  [nomic  ] top-1:    100  sim=0.579  Transakcijski računi u bankama
  [qwen3s ] top-1:   1042  sim=0.519  Escrow račun
  [qwen3m ] top-1:   1050  sim=0.585  Glavna devizna blagajna

QUERY: 'What is the account description for 020?'
  [nomic  ] top-1:   0201  sim=0.623  Poljoprivredno zemljište
  [qwen3s ] top-1:   0201  sim=0.527  Poljoprivredno zemljište
  [qwen3m ] top-1:   1020  sim=0.541  Glavna blagajna (uključivo i plemenitih 

✓ Corpus B ready
Next: 04_evaluate.ipynb
